# Task 4 output profile

Checks that every variant row in `output/*.csv` is unique, and lists any value that repeats.

In [1]:
import csv, glob
from collections import Counter

data = {}
for path in sorted(glob.glob("output/*.csv")):
    with open(path, encoding="utf-8") as f:
        data[path] = list(csv.DictReader(f))
list(data)

['output\\task4-lt-commented.csv', 'output\\task4-lt.csv']

In [2]:
keys = {
    # Files written before the variant_id column existed: the same ID is in the URL.
    "variant_id": lambda r: r.get("variant_id") or r["url"].split("variant=")[-1],
    "url": lambda r: r["url"],
    "product + variant name": lambda r: (r["product_name"], r["variant_name"]),
}
for path, rows in data.items():
    print(f"{path}: {len(rows)} rows, {len({r['product_id'] for r in rows})} products")
    for label, key in keys.items():
        counts = Counter(key(r) for r in rows)
        repeated = {k: c for k, c in counts.items() if c > 1}
        status = "all unique" if not repeated else f"{len(repeated)} repeated"
        print(f"   {label:24} {len(counts):>4} distinct  -> {status}")
        for k, c in list(repeated.items())[:10]:
            print(f"      x{c}: {k}")

output\task4-lt-commented.csv: 321 rows, 95 products
   variant_id                321 distinct  -> all unique
   url                       321 distinct  -> all unique
   product + variant name    321 distinct  -> all unique
output\task4-lt.csv: 321 rows, 95 products
   variant_id                321 distinct  -> all unique
   url                       321 distinct  -> all unique
   product + variant name    321 distinct  -> all unique


In [3]:
# Empty values per column.
for path, rows in data.items():
    empty = {col: sum(not r[col] for r in rows) for col in rows[0]}
    print(path, {col: n for col, n in empty.items() if n})

output\task4-lt-commented.csv {'variant_name': 57, 'description': 1, 'price': 4, 'discounted_price': 293}
output\task4-lt.csv {'variant_name': 57, 'description': 1, 'price': 4, 'discounted_price': 293}


In [ ]:
# Every row should have a price, or (build-your-own sets) a discount percentage instead.
for path, rows in data.items():
    if "flexible_discount_percent" not in rows[0]:
        print(f"{path}: no flexible_discount_percent column (older run), skipped")
        continue
    priced = sum(bool(r["price"]) for r in rows)
    percent_only = [r for r in rows if not r["price"] and r["flexible_discount_percent"]]
    neither = [r for r in rows if not r["price"] and not r["flexible_discount_percent"]]
    print(f"{path}: {priced} with a price, {len(percent_only)} with a discount percentage only, {len(neither)} with neither")
    for r in percent_only:
        print(f"   percentage only: {r['product_name']} ({r['flexible_discount_percent']}%)")
    for r in neither:
        print(f"   NEITHER: {r['product_name']} {r['variant_name']} -> {r['url']}")

In [4]:
# Products with the most variants, and how many variants are out of stock.
for path, rows in data.items():
    per_product = Counter(r["product_name"] for r in rows)
    out = sum(r["in_stock"] == "False" for r in rows)
    print(f"{path}: {out} variants out of stock | most variants: {per_product.most_common(5)}")

output\task4-lt-commented.csv: 29 variants out of stock | most variants: [('Stretch Balm Concealer', 32), ('Stretch Fluid Foundation', 32), ('Cloud Paint', 15), ('Balm Dotcom', 12), ('Perfecting Skin Tint', 12)]
output\task4-lt.csv: 29 variants out of stock | most variants: [('Stretch Balm Concealer', 32), ('Stretch Fluid Foundation', 32), ('Cloud Paint', 15), ('Balm Dotcom', 12), ('Perfecting Skin Tint', 12)]
